# NEPSE-Impact-500: XLM-R and FinBERT Baselines

This notebook uses one frozen balanced 70/15/15 split. Exact and
near-duplicate groups stay together. It trains:

1. XLM-R relevance on all 500 records.
2. XLM-R impact direction on relevant records only.
3. ProsusAI/finbert impact direction on the English relevant subset.

In [ ]:
!pip install -q 'transformers>=4.51,<5' 'datasets>=3.2,<4'   'accelerate>=1.2,<2' 'scikit-learn>=1.5,<2'   'matplotlib>=3.9,<4' 'seaborn>=0.13,<1' 'tqdm>=4.66,<5'

## 1. Load adjudicated data and freeze the common split

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT = Path(os.getenv("MARKET_GYAN_PROJECT", "/content/marketGyan"))
DATA = PROJECT / "data/processed"
SPLITS = DATA / "splits"
OUTPUTS = PROJECT / "outputs"
sys.path.insert(0, str(PROJECT))

from market_gyan.dataset import (
    balanced_group_split,
    compact_qwen_label,
    dataset_readiness,
    read_jsonl,
    split_manifest,
    validate_dataset,
    write_jsonl,
)

gold_path = DATA / "nepse-impact-500.jsonl"
rows = read_jsonl(gold_path)
issues = validate_dataset(rows)
gate = dataset_readiness(rows)
print(json.dumps(gate, indent=2, ensure_ascii=False))
assert not issues, issues[:3]
assert gate["ready"], gate["errors"]

SPLITS.mkdir(parents=True, exist_ok=True)
manifest_path = SPLITS / "manifest.json"
if not manifest_path.exists():
    frozen = balanced_group_split(rows)
    for name, values in frozen.items():
        write_jsonl(SPLITS / f"{name}.jsonl", values)
    manifest_path.write_text(
        json.dumps(
            split_manifest(frozen, strategy="balanced"),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert {item["id"] for item in manifest["assignments"]} == {row["id"] for row in rows}
group_splits = {}
for item in manifest["assignments"]:
    previous = group_splits.setdefault(item["duplicateGroupId"], item["split"])
    assert previous == item["split"], "Near-duplicate group crosses split boundaries"
print(manifest["counts"], manifest["sha256"])

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def plot_counter(axis, counter, title, color, rotate=False):
    items = sorted(counter.items(), key=lambda item: str(item[0]))
    if not items:
        axis.text(0.5, 0.5, "No records", ha="center", va="center")
        axis.set_xticks([])
    else:
        labels, values = zip(*items)
        bars = axis.bar(list(labels), list(values), color=color)
        axis.bar_label(bars, padding=2, fontsize=8)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.2)
    if rotate:
        axis.tick_params(axis="x", rotation=65)

relevance = Counter(row["gold"]["relevance"] for row in rows)
languages = Counter(row["gold"]["language"] for row in rows)
events = Counter(row["gold"]["eventType"] for row in rows)
directions = Counter(
    row["gold"]["impactDirection"]
    for row in rows if row["gold"]["relevance"] != "not_relevant"
)

OUTPUTS.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_counter(axes[0, 0], relevance, "Relevance", "#2563eb")
plot_counter(axes[0, 1], languages, "Language", "#0f766e")
plot_counter(axes[1, 0], events, "Event type", "#7c3aed", rotate=True)
plot_counter(axes[1, 1], directions, "Relevant-record direction", "#dc2626")
plt.tight_layout()
plt.savefig(OUTPUTS / "nepse_impact_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 2. Load the frozen rows

In [ ]:
train_rows = read_jsonl(SPLITS / "train.jsonl")
validation_rows = read_jsonl(SPLITS / "validation.jsonl")
test_rows = read_jsonl(SPLITS / "test.jsonl")
print(len(train_rows), len(validation_rows), len(test_rows))

## 3. Small reusable trainer for the three classifier runs

In [ ]:
import shutil
import numpy as np
import torch
from collections import Counter
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from market_gyan.direction_training import (
    class_weights as compute_class_weights,
    evaluate_with_bias,
    label_counts,
    merge_direction_label,
    oversample_rows,
    oversampling_summary,
    tune_logit_bias,
)

def train_classifier(
    name,
    model_name,
    task,
    labels,
    english_only=False,
    class_weight_scheme="inverse",
    neutral_weight_boost=1.0,
    loss="ce",
    focal_gamma=2.0,
    oversample_direction=False,
    neutral_oversample_factor=4,
    minority_oversample_factor=2,
    merge_neutral_uncertain=False,
    tune_bias=False,
    metric_for_best_model="macro_f1",
):
    is_direction = task == "direction"
    merge = merge_neutral_uncertain and is_direction
    label_to_id = {label: index for index, label in enumerate(labels)}
    id_to_label = {index: label for label, index in label_to_id.items()}
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def direction_of(row):
        return merge_direction_label(
            row["gold"]["impactDirection"],
            merge_neutral_uncertain=merge,
        )

    def target_label(row):
        if task == "relevance":
            return row["gold"]["relevance"]
        return direction_of(row)

    def keep(row):
        if english_only and row["gold"]["language"] != "en":
            return False
        if is_direction and row["gold"]["relevance"] == "not_relevant":
            return False
        return target_label(row) in labels

    def prepare(values, oversample=False):
        filtered = [row for row in values if keep(row)]
        if oversample and is_direction and oversample_direction:
            filtered = oversample_rows(
                filtered,
                direction_of,
                neutral_factor=neutral_oversample_factor,
                minority_factor=minority_oversample_factor,
            )
        dataset = Dataset.from_list([{
            "id": row["id"],
            "text": row["title"] + "\n" + row["excerpt"],
            "label": label_to_id[target_label(row)],
        } for row in filtered])
        return dataset.map(
            lambda batch: tokenizer(
                batch["text"], truncation=True, max_length=512
            ),
            batched=True,
        )

    train_data = prepare(train_rows, oversample=True)
    validation_data = prepare(validation_rows)
    test_data = prepare(test_rows)
    print(
        f"Training {name}: train={len(train_data)}, "
        f"validation={len(validation_data)}, test={len(test_data)}, "
        f"labels={labels}"
    )
    if is_direction and oversample_direction:
        base_rows = [row for row in train_rows if keep(row)]
        print(json.dumps(
            oversampling_summary(
                base_rows,
                direction_of,
                neutral_factor=neutral_oversample_factor,
                minority_factor=minority_oversample_factor,
            ),
            indent=2,
        ))
    # Class weights from the pre-oversampling training label distribution.
    base_counts = label_counts(
        [target_label(row) for row in train_rows if keep(row)],
        labels,
    )
    boosts = (
        {"neutral": neutral_weight_boost}
        if is_direction and neutral_weight_boost != 1.0 and "neutral" in label_to_id
        else None
    )
    weight_values = compute_class_weights(
        base_counts, labels, scheme=class_weight_scheme, boosts=boosts
    )
    weights = torch.tensor(weight_values, dtype=torch.float)
    print("class weights:", dict(zip(labels, [round(w, 3) for w in weight_values])))
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id_to_label,
        label2id=label_to_id,
        ignore_mismatched_sizes=True,
    )

    use_focal = loss == "focal"

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            expected = inputs.pop("labels")
            output = model(**inputs)
            logits = output.logits
            weight = weights.to(logits.device)
            if use_focal:
                # Focal loss down-weights easy majority examples so the scarce
                # neutral rows dominate the gradient signal.
                log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
                gathered = log_probs.gather(1, expected.unsqueeze(1)).squeeze(1)
                probs = gathered.exp()
                sample_weight = weight[expected]
                loss_value = (
                    -((1.0 - probs) ** focal_gamma) * gathered * sample_weight
                ).mean()
            else:
                loss_value = torch.nn.functional.cross_entropy(
                    logits, expected, weight=weight
                )
            return (loss_value, output) if return_outputs else loss_value

    def compute_metrics(prediction):
        predicted = prediction.predictions.argmax(axis=-1)
        report = classification_report(
            prediction.label_ids,
            predicted,
            labels=list(range(len(labels))),
            target_names=labels,
            output_dict=True,
            zero_division=0,
        )
        metrics = {"macro_f1": report["macro avg"]["f1-score"]}
        for label in labels:
            metrics[f"{label}_f1"] = report[label]["f1-score"]
        return metrics

    output_dir = OUTPUTS / name
    arguments = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=8,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model=metric_for_best_model,
        report_to=[],
        seed=42,
        disable_tqdm=False,
        logging_strategy="steps",
        logging_steps=10,
    )
    trainer = WeightedTrainer(
        model=model,
        args=arguments,
        train_dataset=train_data,
        eval_dataset=validation_data,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    checkpoints = sorted(output_dir.glob("checkpoint-*")) if output_dir.exists() else []
    trainer.train(resume_from_checkpoint=str(checkpoints[-1]) if checkpoints else None)
    result = trainer.predict(test_data)
    truth = result.label_ids
    predicted = result.predictions.argmax(axis=-1)

    # Post-hoc per-class logit-bias tuning: fit on validation, apply to test so
    # a directionally-aware model that never wins neutral at argmax can recover
    # its recall without retraining.
    logit_bias = [0.0] * len(labels)
    if is_direction and tune_bias:
        validation_logits = trainer.predict(validation_data)
        logit_bias, tuned_val_f1, _ = tune_logit_bias(
            validation_logits.predictions.tolist(),
            validation_logits.label_ids.tolist(),
            labels,
            present_only=True,
        )
        biased = np.asarray(result.predictions) + np.asarray(logit_bias)
        predicted = biased.argmax(axis=-1)
        print("tuned logit bias:", dict(zip(labels, [round(b, 2) for b in logit_bias])))
        print("validation macro-F1 after bias tuning:", round(tuned_val_f1, 4))

    report = classification_report(
        truth, predicted, labels=list(range(len(labels))),
        target_names=labels, output_dict=True, zero_division=0
    )
    predictions = [{
        "id": test_data[index]["id"],
        "expected": id_to_label[int(truth[index])],
        "predicted": id_to_label[int(predicted[index])],
    } for index in range(len(test_data))]
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "metrics.json").write_text(json.dumps(report, indent=2))
    (output_dir / "predictions.json").write_text(json.dumps(predictions, indent=2))
    if is_direction:
        (output_dir / "direction_config.json").write_text(json.dumps({
            "classWeightScheme": class_weight_scheme,
            "neutralWeightBoost": neutral_weight_boost,
            "loss": loss,
            "focalGamma": focal_gamma if use_focal else None,
            "oversampleDirection": oversample_direction,
            "mergeNeutralUncertain": merge,
            "tuneBias": tune_bias,
            "logitBias": dict(zip(labels, logit_bias)),
            "metricForBestModel": metric_for_best_model,
        }, indent=2))
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    for checkpoint in output_dir.glob("checkpoint-*"):
        shutil.rmtree(checkpoint, ignore_errors=True)
    return trainer, report, truth, predicted, labels, output_dir

## 4. Train XLM-R relevance

In [ ]:
set_seed(42)
relevance_run = train_classifier(
    "xlmr-relevance",
    "xlm-roberta-base",
    "relevance",
    ["direct", "indirect", "not_relevant"],
)

## 5. Train XLM-R direction on relevant records

The impact-direction task has a severe minority-class problem: the frozen split
has only ~14 `neutral` training rows against 122 `uncertain` / 83 `bullish` /
62 `bearish`, so plain argmax over a weighted softmax almost never predicts
`neutral` and its F1 collapses. This run turns on the rare-class levers from
`market_gyan.direction_training`:

* `effective_number` class weights (gentler than raw inverse frequency for
  ultra-rare classes) with an extra `neutral` boost,
* `focal` loss to down-weight easy majority examples,
* minority oversampling (`neutral` x4, `bearish`/`uncertain` x2),
* `neutral_f1` model selection, and
* post-hoc logit-bias tuning fit on validation and applied to test.

The bias and config are saved to `direction_config.json` next to the metrics.

In [ ]:
direction_run = train_classifier(
    "xlmr-direction",
    "xlm-roberta-base",
    "direction",
    ["bullish", "bearish", "neutral", "uncertain"],
    class_weight_scheme="effective_number",
    neutral_weight_boost=2.0,
    loss="focal",
    focal_gamma=2.0,
    oversample_direction=True,
    neutral_oversample_factor=4,
    minority_oversample_factor=2,
    tune_bias=True,
    metric_for_best_model="neutral_f1",
)

## 6. Train the English-only FinBERT baseline

In [ ]:
finbert_run = train_classifier(
    "finbert-english-direction",
    "ProsusAI/finbert",
    "direction",
    ["bullish", "bearish", "neutral"],
    english_only=True,
)

## 7. Plot confusion matrices, class F1, and loss

In [ ]:
import seaborn as sns

for name, run in {
    "xlmr_relevance": relevance_run,
    "xlmr_direction": direction_run,
    "finbert_direction": finbert_run,
}.items():
    trainer, report, truth, predicted, labels, output_dir = run
    matrix = confusion_matrix(truth, predicted, labels=list(range(len(labels))))
    history = trainer.state.log_history
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0],
        xticklabels=labels, yticklabels=labels
    )
    axes[0].set_title(f"{name} confusion matrix")
    bars = axes[1].bar(list(labels), [report[label]["f1-score"] for label in labels])
    axes[1].bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Per-class F1")
    train_steps = [row["step"] for row in history if "loss" in row]
    train_loss = [row["loss"] for row in history if "loss" in row]
    eval_steps = [row["step"] for row in history if "eval_loss" in row]
    eval_loss = [row["eval_loss"] for row in history if "eval_loss" in row]
    if train_steps:
        axes[2].plot(train_steps, train_loss, label="train")
    if eval_steps:
        axes[2].plot(eval_steps, eval_loss, label="validation")
    axes[2].set_title("Loss")
    axes[2].legend()
    plt.tight_layout()
    plt.savefig(output_dir / "results.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 8. Archive the three baselines

In [ ]:
import shutil
import tempfile
from pathlib import Path

archive_base = PROJECT / "nepse-impact-baselines"
archive_path = archive_base.with_suffix(".zip")
if archive_path.exists():
    archive_path.unlink()

with tempfile.TemporaryDirectory() as directory:
    staging = Path(directory) / "nepse-impact-baselines"
    staging.mkdir(parents=True, exist_ok=True)
    for name in ("xlmr-relevance", "xlmr-direction", "finbert-direction"):
        source = OUTPUTS / name
        if not source.exists():
            continue
        target = staging / name
        shutil.copytree(source, target)
        for checkpoint in target.glob("checkpoint-*"):
            shutil.rmtree(checkpoint, ignore_errors=True)
    archive = shutil.make_archive(
        str(archive_base),
        "zip",
        staging,
    )
print(archive)
# Colab: from google.colab import files; files.download(archive)